In [ ]:
# ============================================================
# acrobot_reliable_v4_stable.py
# ------------------------------------------------------------
# Acrobot 전용 more-stable recurrent PPO
# - notebook-safe argparse
# - conservative PPO
# - smoother curriculum
# - less aggressive shaping
# - best checkpoint restore
# - improved collapse handling
# ============================================================

import os
import gc
import csv
import json
import math
import signal
import random
import argparse
from copy import deepcopy
from dataclasses import dataclass, asdict
from typing import List, Optional, Tuple

import numpy as np
import gymnasium as gym

import torch
import torch.nn as nn
from torch.distributions import Categorical


# ============================================================
# Global
# ============================================================
STOP_REQUESTED = False


def _signal_handler(signum, frame):
    global STOP_REQUESTED
    STOP_REQUESTED = True
    print(f"\n[signal] received {signum}, stop at safe point.")


signal.signal(signal.SIGINT, _signal_handler)
signal.signal(signal.SIGTERM, _signal_handler)


# ============================================================
# Utils
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def cleanup(device: torch.device):
    try:
        gc.collect()
        if device.type == "cuda":
            torch.cuda.synchronize(device)
            torch.cuda.empty_cache()
            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass
    except Exception:
        pass


def linear_interp(a: float, b: float, t01: float) -> float:
    t01 = max(0.0, min(1.0, float(t01)))
    return a + (b - a) * t01


def cosine_decay(start: float, end: float, t01: float) -> float:
    t01 = max(0.0, min(1.0, float(t01)))
    c = 0.5 * (1.0 + math.cos(math.pi * t01))
    return end + (start - end) * c


def gpu_mem_str(device: torch.device) -> str:
    if device.type != "cuda":
        return "cpu"
    alloc = torch.cuda.memory_allocated(device) / (1024 ** 2)
    reserved = torch.cuda.memory_reserved(device) / (1024 ** 2)
    return f"alloc={alloc:.1f}MB reserved={reserved:.1f}MB"


# ============================================================
# Config
# ============================================================
@dataclass
class Config:
    name: str = "acrobot_reliable_v4_stable"
    env_id: str = "Acrobot-v1"
    partial_indices: Optional[List[int]] = None

    # smoother curriculum
    flicker_prob_start: float = 0.0
    flicker_prob_end: float = 0.008
    noise_std_start: float = 0.0
    noise_std_end: float = 0.00015
    curriculum_start_ratio: float = 0.15   # first 15% mostly clean
    curriculum_end_ratio: float = 0.90     # ramp until 90%

    max_steps: int = 500
    recommended_episodes: int = 24000
    solve_score: float = -100.0

    # more conservative PPO
    lr: float = 8e-5
    lr_end_scale: float = 0.35
    gamma: float = 0.995
    gae_lambda: float = 0.98
    ppo_clip: float = 0.12
    value_clip: float = 0.12
    ppo_epochs: int = 6
    minibatch_tokens: int = 2048
    steps_per_update: int = 2048
    value_coef: float = 0.7
    max_grad_norm: float = 0.35
    target_kl: float = 0.008

    ent_start: float = 0.030
    ent_end: float = 0.005

    mlp_hidden: int = 256
    gru_hidden: int = 192

    eval_every: int = 200
    eval_episodes: int = 12
    save_every: int = 200
    log_every: int = 10

    ckpt_dir: str = "./checkpoints_acrobot_v4"
    raw_csv: str = "./acrobot_v4_raw.csv"
    summary_json: str = "./acrobot_v4_summary.json"

    # collapse / anchor
    spike_threshold: float = 6.0
    collapse_threshold: float = 12.0
    anchor_blend: float = 0.20
    warmup_episodes: int = 800
    collapse_restore_patience: int = 2

    # shaping
    potential_scale: float = 5.0
    vel_penalty_scale: float = 0.015
    y_gain_scale: float = 1.2
    terminal_bonus: float = 8.0


CFG = Config(partial_indices=[0, 1, 2, 3, 4, 5])


# ============================================================
# RunningMeanStd
# ============================================================
class RunningMeanStd:
    def __init__(self, shape):
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var = np.ones(shape, dtype=np.float64)
        self.count = 1e-4

    def update(self, x: np.ndarray):
        x = np.asarray(x, dtype=np.float64)
        if x.ndim == 1:
            x = x[None, :]

        batch_mean = np.mean(x, axis=0)
        batch_var = np.var(x, axis=0)
        batch_count = x.shape[0]

        delta = batch_mean - self.mean
        total_count = self.count + batch_count

        new_mean = self.mean + delta * batch_count / total_count
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + (delta ** 2) * self.count * batch_count / total_count
        new_var = M2 / total_count

        self.mean = new_mean
        self.var = np.maximum(new_var, 1e-12)
        self.count = total_count

    def normalize(self, x: np.ndarray) -> np.ndarray:
        x = np.asarray(x, dtype=np.float32)
        z = (x - self.mean) / np.sqrt(self.var + 1e-8)
        return np.clip(z, -10.0, 10.0).astype(np.float32)

    def state_dict(self):
        return {
            "mean": self.mean.tolist(),
            "var": self.var.tolist(),
            "count": float(self.count),
        }

    def load_state_dict(self, sd):
        self.mean = np.asarray(sd["mean"], dtype=np.float64)
        self.var = np.asarray(sd["var"], dtype=np.float64)
        self.count = float(sd["count"])


# ============================================================
# POMDP Wrapper
# ============================================================
class ScheduledPartialObsWrapper(gym.ObservationWrapper):
    def __init__(self, env, partial_indices, flicker_prob_start, flicker_prob_end, noise_std_start, noise_std_end):
        super().__init__(env)
        self.partial_indices = partial_indices
        self.flicker_prob_start = float(flicker_prob_start)
        self.flicker_prob_end = float(flicker_prob_end)
        self.noise_std_start = float(noise_std_start)
        self.noise_std_end = float(noise_std_end)
        self.progress = 0.0

        low = env.observation_space.low
        high = env.observation_space.high

        if self.partial_indices is None:
            self.partial_indices = list(range(int(np.prod(low.shape))))

        low2 = low[self.partial_indices].astype(np.float32)
        high2 = high[self.partial_indices].astype(np.float32)

        self.observation_space = gym.spaces.Box(
            low=low2,
            high=high2,
            shape=(len(self.partial_indices),),
            dtype=np.float32,
        )

    def set_progress(self, progress01: float):
        self.progress = max(0.0, min(1.0, float(progress01)))

    def get_current_params(self):
        flicker = linear_interp(self.flicker_prob_start, self.flicker_prob_end, self.progress)
        noise = linear_interp(self.noise_std_start, self.noise_std_end, self.progress)
        return flicker, noise

    def observation(self, obs):
        obs = np.asarray(obs, dtype=np.float32)[self.partial_indices]
        flicker, noise = self.get_current_params()

        if flicker > 0.0 and np.random.rand() < flicker:
            obs = np.zeros_like(obs, dtype=np.float32)

        if noise > 0.0:
            obs = obs + np.random.normal(0.0, noise, size=obs.shape).astype(np.float32)

        return obs.astype(np.float32)


# ============================================================
# Reward shaping
# ============================================================
class AcrobotShaperV4:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def tip_height(self, obs):
        c1, s1, c2, s2 = float(obs[0]), float(obs[1]), float(obs[2]), float(obs[3])
        theta1 = math.atan2(s1, c1)
        theta2 = math.atan2(s2, c2)
        y = -math.cos(theta1) - math.cos(theta1 + theta2)
        return y

    def potential(self, obs):
        y = self.tip_height(obs)
        d1 = float(obs[4]) if len(obs) > 4 else 0.0
        d2 = float(obs[5]) if len(obs) > 5 else 0.0
        vel_pen = self.cfg.vel_penalty_scale * (abs(d1) + abs(d2))
        return self.cfg.potential_scale * y - vel_pen

    def train_reward(self, obs, next_obs, raw_reward, terminated):
        # less aggressive shaping than v3
        shaped = float(raw_reward) + self.cfg.gamma * self.potential(next_obs) - self.potential(obs)
        y_gain = self.tip_height(next_obs) - self.tip_height(obs)
        shaped += self.cfg.y_gain_scale * y_gain

        if terminated:
            shaped += self.cfg.terminal_bonus

        # light clipping for stability
        return float(np.clip(shaped, -5.0, 5.0))


# ============================================================
# Model
# ============================================================
class RecurrentActorCritic(nn.Module):
    def __init__(self, obs_dim, action_dim, mlp_hidden, gru_hidden):
        super().__init__()
        self.gru_hidden = gru_hidden

        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, mlp_hidden),
            nn.LayerNorm(mlp_hidden),
            nn.Tanh(),
            nn.Linear(mlp_hidden, mlp_hidden),
            nn.LayerNorm(mlp_hidden),
            nn.Tanh(),
        )
        self.gru = nn.GRU(mlp_hidden, gru_hidden, batch_first=True)
        self.actor = nn.Linear(gru_hidden, action_dim)
        self.critic = nn.Linear(gru_hidden, 1)

        self.apply(self._init)
        nn.init.orthogonal_(self.actor.weight, gain=0.01)
        nn.init.zeros_(self.actor.bias)
        nn.init.orthogonal_(self.critic.weight, gain=1.0)
        nn.init.zeros_(self.critic.bias)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.orthogonal_(m.weight, gain=math.sqrt(2))
            nn.init.zeros_(m.bias)

    def initial_state(self, batch_size, device):
        return torch.zeros(1, batch_size, self.gru_hidden, device=device)

    def forward(self, obs_seq, hidden=None):
        x = self.encoder(obs_seq)
        x, hidden = self.gru(x, hidden)
        logits = self.actor(x)
        values = self.critic(x)
        return logits, values, hidden

    @torch.no_grad()
    def act(self, obs_t, hidden):
        logits, value, hidden = self.forward(obs_t, hidden)
        logits = logits[:, -1, :]
        value = value[:, -1, :]
        dist = Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return int(action.item()), float(log_prob.item()), float(value.item()), hidden


# ============================================================
# RolloutBuffer
# ============================================================
class RolloutBuffer:
    def __init__(self):
        self.obs = []
        self.actions = []
        self.rewards = []
        self.raw_rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.episode_ids = []
        self.bootstrap_values = []

    def add(self, obs, action, reward, raw_reward, done, log_prob, value, episode_id):
        self.obs.append(np.asarray(obs, dtype=np.float32))
        self.actions.append(int(action))
        self.rewards.append(float(reward))
        self.raw_rewards.append(float(raw_reward))
        self.dones.append(bool(done))
        self.log_probs.append(float(log_prob))
        self.values.append(float(value))
        self.episode_ids.append(int(episode_id))

    def clear(self):
        self.__init__()

    def __len__(self):
        return len(self.obs)


# ============================================================
# Helpers
# ============================================================
def build_env(seed=None):
    env = gym.make(CFG.env_id)
    env = ScheduledPartialObsWrapper(
        env,
        CFG.partial_indices,
        CFG.flicker_prob_start,
        CFG.flicker_prob_end,
        CFG.noise_std_start,
        CFG.noise_std_end,
    )
    if seed is not None:
        env.reset(seed=seed)
        env.action_space.seed(seed)
    return env


def compute_curriculum_progress(progress01: float) -> float:
    if progress01 <= CFG.curriculum_start_ratio:
        return 0.0
    if progress01 >= CFG.curriculum_end_ratio:
        return 1.0
    denom = max(1e-8, CFG.curriculum_end_ratio - CFG.curriculum_start_ratio)
    return (progress01 - CFG.curriculum_start_ratio) / denom


def compute_gae(rewards, values, dones, next_value, gamma, gae_lambda):
    n = len(rewards)
    adv = np.zeros(n, dtype=np.float32)
    last_gae = 0.0

    for t in reversed(range(n)):
        if t == n - 1:
            next_nonterminal = 1.0 - float(dones[t])
            next_values = next_value
        else:
            next_nonterminal = 1.0 - float(dones[t])
            next_values = values[t + 1]

        delta = rewards[t] + gamma * next_values * next_nonterminal - values[t]
        last_gae = delta + gamma * gae_lambda * next_nonterminal * last_gae
        adv[t] = last_gae

    ret = adv + values
    return adv, ret


def split_segments_preview(buffer: RolloutBuffer):
    if len(buffer) == 0:
        return []

    segments = []
    start = 0

    for i in range(len(buffer.obs) - 1):
        if buffer.dones[i] or buffer.episode_ids[i + 1] != buffer.episode_ids[i]:
            segments.append(slice(start, i + 1))
            start = i + 1

    if start < len(buffer.obs):
        segments.append(slice(start, len(buffer.obs)))

    return segments


def split_segments(buffer: RolloutBuffer):
    previews = split_segments_preview(buffer)
    return [(sl, buffer.bootstrap_values[idx]) for idx, sl in enumerate(previews)]


def pad_sequences(arrays, pad_value=0.0):
    max_len = max(a.shape[0] for a in arrays)
    padded, masks = [], []

    for a in arrays:
        t = a.shape[0]
        pad_shape = (max_len - t,) + a.shape[1:]
        pad = np.full(pad_shape, pad_value, dtype=a.dtype)
        padded.append(np.concatenate([a, pad], axis=0))
        masks.append(np.concatenate([
            np.ones(t, dtype=np.float32),
            np.zeros(max_len - t, dtype=np.float32)
        ]))

    return np.stack(padded), np.stack(masks)


@torch.no_grad()
def evaluate(model, obs_rms, device, n_episodes=10):
    env = build_env(seed=12345)
    env.set_progress(1.0)
    scores = []

    try:
        for ep in range(n_episodes):
            obs, _ = env.reset(seed=12345 + ep)
            hidden = model.initial_state(1, device)
            done = False
            ep_reward = 0.0
            steps = 0

            while not done and steps < CFG.max_steps:
                obs_n = obs_rms.normalize(obs)
                obs_t = torch.as_tensor(obs_n, dtype=torch.float32, device=device).view(1, 1, -1)
                logits, _, hidden = model(obs_t, hidden)
                action = torch.argmax(Categorical(logits=logits[:, -1, :]).probs, dim=-1)

                obs, reward, terminated, truncated, _ = env.step(int(action.item()))
                done = terminated or truncated
                ep_reward += float(reward)
                steps += 1

            scores.append(ep_reward)
    finally:
        env.close()

    arr = np.asarray(scores, dtype=np.float32)
    return {
        "eval_mean": float(arr.mean()),
        "eval_std": float(arr.std(ddof=0)),
        "eval_min": float(arr.min()),
        "eval_max": float(arr.max()),
    }


def maybe_anchor_blend(model, anchor_model, blend):
    with torch.no_grad():
        msd = model.state_dict()
        asd = anchor_model.state_dict()
        for k in msd.keys():
            msd[k].mul_(1.0 - blend).add_(asd[k], alpha=blend)
        model.load_state_dict(msd)


def ppo_update(model, optimizer, scaler, buffer, device, use_amp, progress01):
    segments = split_segments(buffer)
    if not segments:
        return {"loss": 0.0, "policy": 0.0, "value": 0.0, "entropy": 0.0, "kl": 0.0, "early_stop_kl": 0}

    obs_list, act_list, old_logp_list, adv_list, ret_list, old_val_list = [], [], [], [], [], []

    for sl, bootstrap_value in segments:
        rewards = np.asarray(buffer.rewards[sl], dtype=np.float32)
        values = np.asarray(buffer.values[sl], dtype=np.float32)
        dones = np.asarray(buffer.dones[sl], dtype=np.float32)
        adv, ret = compute_gae(rewards, values, dones, bootstrap_value, CFG.gamma, CFG.gae_lambda)

        obs_list.append(np.asarray(buffer.obs[sl], dtype=np.float32))
        act_list.append(np.asarray(buffer.actions[sl], dtype=np.int64))
        old_logp_list.append(np.asarray(buffer.log_probs[sl], dtype=np.float32))
        adv_list.append(adv)
        ret_list.append(ret.astype(np.float32))
        old_val_list.append(np.asarray(buffer.values[sl], dtype=np.float32))

    all_adv = np.concatenate(adv_list, axis=0)
    adv_mean, adv_std = all_adv.mean(), all_adv.std() + 1e-8
    adv_list = [((a - adv_mean) / adv_std).astype(np.float32) for a in adv_list]

    obs_pad, mask = pad_sequences(obs_list, 0.0)
    act_pad, _ = pad_sequences([a[:, None] for a in act_list], 0)
    old_logp_pad, _ = pad_sequences([a[:, None] for a in old_logp_list], 0.0)
    adv_pad, _ = pad_sequences([a[:, None] for a in adv_list], 0.0)
    ret_pad, _ = pad_sequences([a[:, None] for a in ret_list], 0.0)
    old_val_pad, _ = pad_sequences([a[:, None] for a in old_val_list], 0.0)

    obs_t = torch.as_tensor(obs_pad, dtype=torch.float32, device=device)
    act_t = torch.as_tensor(act_pad.squeeze(-1), dtype=torch.long, device=device)
    old_logp_t = torch.as_tensor(old_logp_pad.squeeze(-1), dtype=torch.float32, device=device)
    adv_t = torch.as_tensor(adv_pad.squeeze(-1), dtype=torch.float32, device=device)
    ret_t = torch.as_tensor(ret_pad.squeeze(-1), dtype=torch.float32, device=device)
    old_val_t = torch.as_tensor(old_val_pad.squeeze(-1), dtype=torch.float32, device=device)
    mask_t = torch.as_tensor(mask, dtype=torch.float32, device=device)

    idxs = np.arange(obs_t.shape[0])
    seqs_per_batch = max(1, CFG.minibatch_tokens // max(1, obs_t.shape[1]))
    ent_coef = cosine_decay(CFG.ent_start, CFG.ent_end, progress01)

    losses, plosses, vlosses, ents, kls = [], [], [], [], []
    early_stop_kl = 0

    for _ in range(CFG.ppo_epochs):
        np.random.shuffle(idxs)
        stop_epoch = False

        for start in range(0, len(idxs), seqs_per_batch):
            bidx = idxs[start:start + seqs_per_batch]

            b_obs = obs_t[bidx]
            b_act = act_t[bidx]
            b_old_logp = old_logp_t[bidx]
            b_adv = adv_t[bidx]
            b_ret = ret_t[bidx]
            b_old_val = old_val_t[bidx]
            b_mask = mask_t[bidx]

            hidden = model.initial_state(b_obs.shape[0], device)

            with torch.amp.autocast("cuda", enabled=use_amp):
                logits, values, _ = model(b_obs, hidden)
                values = values.squeeze(-1)

                dist = Categorical(logits=logits)
                new_logp = dist.log_prob(b_act)
                entropy = dist.entropy()

                ratio = torch.exp(new_logp - b_old_logp)
                surr1 = ratio * b_adv
                surr2 = torch.clamp(ratio, 1.0 - CFG.ppo_clip, 1.0 + CFG.ppo_clip) * b_adv
                policy_loss = -torch.sum(torch.min(surr1, surr2) * b_mask) / (torch.sum(b_mask) + 1e-8)

                value_pred_clipped = b_old_val + torch.clamp(values - b_old_val, -CFG.value_clip, CFG.value_clip)
                value_loss_unclipped = (values - b_ret) ** 2
                value_loss_clipped = (value_pred_clipped - b_ret) ** 2
                value_loss = 0.5 * torch.sum(torch.max(value_loss_unclipped, value_loss_clipped) * b_mask) / (
                    torch.sum(b_mask) + 1e-8
                )

                entropy_mean = torch.sum(entropy * b_mask) / (torch.sum(b_mask) + 1e-8)
                loss = policy_loss + CFG.value_coef * value_loss - ent_coef * entropy_mean

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()

            with torch.no_grad():
                approx_kl = torch.sum((b_old_logp - new_logp) * b_mask) / (torch.sum(b_mask) + 1e-8)

            losses.append(float(loss.item()))
            plosses.append(float(policy_loss.item()))
            vlosses.append(float(value_loss.item()))
            ents.append(float(entropy_mean.item()))
            kls.append(float(approx_kl.item()))

            if float(approx_kl.item()) > CFG.target_kl:
                early_stop_kl += 1
                stop_epoch = True
                break

        if stop_epoch:
            break

    return {
        "loss": float(np.mean(losses)) if losses else 0.0,
        "policy": float(np.mean(plosses)) if plosses else 0.0,
        "value": float(np.mean(vlosses)) if vlosses else 0.0,
        "entropy": float(np.mean(ents)) if ents else 0.0,
        "kl": float(np.mean(kls)) if kls else 0.0,
        "early_stop_kl": early_stop_kl,
    }


def save_checkpoint(
    path,
    episode,
    global_step,
    model,
    optimizer,
    scaler,
    obs_rms,
    raw_train_rewards,
    train_rewards,
    eval_history,
    spike_count,
    collapse_count,
    anchor_model,
    best_eval_mean,
    best_eval_episode,
):
    torch.save({
        "episode": int(episode),
        "global_step": int(global_step),
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "obs_rms": obs_rms.state_dict(),
        "raw_train_rewards": list(raw_train_rewards),
        "train_rewards": list(train_rewards),
        "eval_history": list(eval_history),
        "spike_count": int(spike_count),
        "collapse_count": int(collapse_count),
        "anchor_model": anchor_model.state_dict() if anchor_model is not None else None,
        "best_eval_mean": float(best_eval_mean),
        "best_eval_episode": int(best_eval_episode),
        "config": asdict(CFG),
    }, path)


def load_checkpoint(path, model, optimizer, scaler, obs_rms, anchor_model, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    try:
        scaler.load_state_dict(ckpt["scaler"])
    except Exception:
        pass
    obs_rms.load_state_dict(ckpt["obs_rms"])
    if ckpt.get("anchor_model") is not None:
        anchor_model.load_state_dict(ckpt["anchor_model"])
    return ckpt


# ============================================================
# Main
# ============================================================
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--device", type=str, default=None)
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--episodes", type=int, default=CFG.recommended_episodes)
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--amp", action="store_true")
    args, _ = parser.parse_known_args()

    set_seed(args.seed)
    device = torch.device(args.device if args.device is not None else ("cuda:0" if torch.cuda.is_available() else "cpu"))
    use_amp = bool(args.amp and device.type == "cuda")

    print("=" * 68)
    print("device       =", device)
    print("use_amp      =", use_amp)
    if device.type == "cuda":
        print("cuda_name    =", torch.cuda.get_device_name(device))
        print("cuda_count   =", torch.cuda.device_count())
        props = torch.cuda.get_device_properties(device)
        print("total_memory =", round(props.total_memory / (1024 ** 3), 2), "GB")
    else:
        print("cuda_name    = CPU mode")
    print("=" * 68)

    env = build_env(seed=args.seed)
    shaper = AcrobotShaperV4(CFG)

    obs_dim = int(np.prod(env.observation_space.shape))
    action_dim = int(env.action_space.n)

    model = RecurrentActorCritic(obs_dim, action_dim, CFG.mlp_hidden, CFG.gru_hidden).to(device)
    anchor_model = RecurrentActorCritic(obs_dim, action_dim, CFG.mlp_hidden, CFG.gru_hidden).to(device)
    anchor_model.load_state_dict(model.state_dict())
    anchor_model.eval()

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, eps=1e-5, weight_decay=1e-5)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    obs_rms = RunningMeanStd((obs_dim,))
    buffer = RolloutBuffer()

    ensure_dir(CFG.ckpt_dir)
    latest_path = os.path.join(CFG.ckpt_dir, f"{CFG.name}_seed{args.seed}_latest.pt")
    best_path = os.path.join(CFG.ckpt_dir, f"{CFG.name}_seed{args.seed}_best.pt")

    raw_train_rewards, train_rewards, eval_history = [], [], []
    episode_idx, global_step = 0, 0
    best_anchor_eval = -1e18
    best_eval_mean = -1e18
    best_eval_episode = -1
    spike_count, collapse_count = 0, 0
    bad_eval_streak = 0

    if args.resume and os.path.exists(latest_path):
        ckpt = load_checkpoint(latest_path, model, optimizer, scaler, obs_rms, anchor_model, device)
        raw_train_rewards = list(ckpt.get("raw_train_rewards", []))
        train_rewards = list(ckpt.get("train_rewards", []))
        eval_history = list(ckpt.get("eval_history", []))
        spike_count = int(ckpt.get("spike_count", 0))
        collapse_count = int(ckpt.get("collapse_count", 0))
        episode_idx = int(ckpt.get("episode", -1)) + 1
        global_step = int(ckpt.get("global_step", 0))
        best_eval_mean = float(ckpt.get("best_eval_mean", -1e18))
        best_eval_episode = int(ckpt.get("best_eval_episode", -1))
        if eval_history:
            best_anchor_eval = max(x["eval_mean"] for x in eval_history)
        print(f"[resume] start_episode={episode_idx}")

    try:
        obs, _ = env.reset(seed=args.seed + 123)
        hidden = model.initial_state(1, device)
        ep_raw_reward, ep_train_reward, ep_steps = 0.0, 0.0, 0
        done = False

        while episode_idx < args.episodes:
            progress01 = episode_idx / max(1, args.episodes - 1)
            env_progress = compute_curriculum_progress(progress01)
            env.set_progress(env_progress)

            lr_now = cosine_decay(CFG.lr, CFG.lr * CFG.lr_end_scale, progress01)
            for g in optimizer.param_groups:
                g["lr"] = lr_now

            obs_rms.update(np.asarray(obs, dtype=np.float32)[None, :])
            obs_n = obs_rms.normalize(obs)
            obs_t = torch.as_tensor(obs_n, dtype=torch.float32, device=device).view(1, 1, -1)

            action, log_prob, value, hidden = model.act(obs_t, hidden)
            next_obs, raw_reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            train_reward = shaper.train_reward(obs, next_obs, raw_reward, terminated)
            buffer.add(obs_n, action, train_reward, raw_reward, done, log_prob, value, episode_idx)

            obs = next_obs
            ep_raw_reward += float(raw_reward)
            ep_train_reward += float(train_reward)
            ep_steps += 1
            global_step += 1

            if done or ep_steps >= CFG.max_steps:
                buffer.bootstrap_values.append(0.0)
                raw_train_rewards.append(ep_raw_reward)
                train_rewards.append(ep_train_reward)

                if episode_idx % CFG.log_every == 0:
                    last20 = raw_train_rewards[-20:] if len(raw_train_rewards) >= 20 else raw_train_rewards
                    flicker_now, noise_now = env.get_current_params()
                    print(
                        f"[train] ep={episode_idx:05d}/{args.episodes} "
                        f"({100.0 * episode_idx / max(1, args.episodes):5.1f}%) "
                        f"raw={ep_raw_reward:8.3f} train={ep_train_reward:8.3f} "
                        f"recent20={np.mean(last20):8.3f} "
                        f"flicker={flicker_now:.4f} noise={noise_now:.5f} "
                        f"lr={lr_now:.7f} gpu={gpu_mem_str(device)}"
                    )

                if ((episode_idx + 1) % CFG.eval_every) == 0 or (episode_idx + 1) == args.episodes:
                    eval_result = evaluate(model, obs_rms, device, CFG.eval_episodes)
                    eval_result["eval_at_episode"] = episode_idx + 1
                    eval_history.append(eval_result)

                    print(
                        f"[eval] ep={episode_idx+1} mean={eval_result['eval_mean']:.3f} "
                        f"std={eval_result['eval_std']:.3f} "
                        f"min={eval_result['eval_min']:.3f} max={eval_result['eval_max']:.3f}"
                    )

                    # best model save
                    if eval_result["eval_mean"] > best_eval_mean:
                        best_eval_mean = eval_result["eval_mean"]
                        best_eval_episode = episode_idx + 1
                        torch.save({
                            "model": model.state_dict(),
                            "obs_rms": obs_rms.state_dict(),
                            "best_eval_mean": best_eval_mean,
                            "best_eval_episode": best_eval_episode,
                            "config": asdict(CFG),
                        }, best_path)
                        print(f"[best] saved best checkpoint @ ep={best_eval_episode} eval_mean={best_eval_mean:.3f}")
                        bad_eval_streak = 0
                    else:
                        bad_eval_streak += 1

                    # anchor updates
                    if episode_idx >= CFG.warmup_episodes and eval_result["eval_mean"] > best_anchor_eval + CFG.spike_threshold:
                        best_anchor_eval = eval_result["eval_mean"]
                        anchor_model.load_state_dict(deepcopy(model.state_dict()))
                        spike_count += 1
                        print(f"[SPIKE] new anchor saved | spike_count={spike_count} | anchor_eval={best_anchor_eval:.3f}")

                    # collapse handling
                    if episode_idx >= CFG.warmup_episodes and best_anchor_eval > -1e17:
                        if eval_result["eval_mean"] < best_anchor_eval - CFG.collapse_threshold:
                            collapse_count += 1
                            maybe_anchor_blend(model, anchor_model, CFG.anchor_blend)
                            print(f"[COLLAPSE] anchor blend applied | collapse_count={collapse_count}")

                            # if repeated bad evals and best ckpt exists, softly restore best
                            if bad_eval_streak >= CFG.collapse_restore_patience and os.path.exists(best_path):
                                best_ckpt = torch.load(best_path, map_location=device)
                                model.load_state_dict(best_ckpt["model"])
                                obs_rms.load_state_dict(best_ckpt["obs_rms"])
                                bad_eval_streak = 0
                                print(f"[RESTORE] loaded best checkpoint from ep={best_ckpt['best_eval_episode']}")

                if ((episode_idx + 1) % CFG.save_every) == 0 or STOP_REQUESTED:
                    save_checkpoint(
                        latest_path,
                        episode_idx,
                        global_step,
                        model,
                        optimizer,
                        scaler,
                        obs_rms,
                        raw_train_rewards,
                        train_rewards,
                        eval_history,
                        spike_count,
                        collapse_count,
                        anchor_model,
                        best_eval_mean,
                        best_eval_episode,
                    )
                    print(f"[ckpt] saved {latest_path}")

                episode_idx += 1
                if STOP_REQUESTED:
                    break

                obs, _ = env.reset(seed=args.seed * 100000 + episode_idx)
                hidden = model.initial_state(1, device)
                ep_raw_reward, ep_train_reward, ep_steps = 0.0, 0.0, 0

            if len(buffer) >= CFG.steps_per_update or (STOP_REQUESTED and len(buffer) > 0):
                needed = len(split_segments_preview(buffer))
                while len(buffer.bootstrap_values) < needed:
                    obs_n = obs_rms.normalize(obs)
                    obs_t = torch.as_tensor(obs_n, dtype=torch.float32, device=device).view(1, 1, -1)
                    with torch.no_grad():
                        _, value_t, _ = model(obs_t, hidden)
                        bootstrap_value = float(value_t[:, -1, :].item()) if not done else 0.0
                    buffer.bootstrap_values.append(bootstrap_value)

                stats = ppo_update(model, optimizer, scaler, buffer, device, use_amp, progress01)
                print(
                    f"[update] loss={stats['loss']:.4f} policy={stats['policy']:.4f} "
                    f"value={stats['value']:.4f} entropy={stats['entropy']:.4f} "
                    f"kl={stats['kl']:.4f} early_stop_kl={stats['early_stop_kl']} "
                    f"gpu={gpu_mem_str(device)}"
                )
                buffer.clear()

                if STOP_REQUESTED:
                    break

        # save latest before final report
        save_checkpoint(
            latest_path,
            max(0, episode_idx - 1),
            global_step,
            model,
            optimizer,
            scaler,
            obs_rms,
            raw_train_rewards,
            train_rewards,
            eval_history,
            spike_count,
            collapse_count,
            anchor_model,
            best_eval_mean,
            best_eval_episode,
        )

        # final result should use BEST model, not last model
        final_model = model
        final_obs_rms = obs_rms

        if os.path.exists(best_path):
            best_ckpt = torch.load(best_path, map_location=device)
            final_model = RecurrentActorCritic(obs_dim, action_dim, CFG.mlp_hidden, CFG.gru_hidden).to(device)
            final_model.load_state_dict(best_ckpt["model"])
            final_model.eval()

            final_obs_rms = RunningMeanStd((obs_dim,))
            final_obs_rms.load_state_dict(best_ckpt["obs_rms"])

        final_eval = evaluate(final_model, final_obs_rms, device, max(CFG.eval_episodes, 20))

        tail100 = raw_train_rewards[-100:] if len(raw_train_rewards) >= 100 else raw_train_rewards
        result = {
            "preset": CFG.name,
            "seed": args.seed,
            "episodes_completed": len(raw_train_rewards),
            "last100_mean": float(np.mean(tail100)) if tail100 else 0.0,
            "best_reward": float(np.max(raw_train_rewards)) if raw_train_rewards else 0.0,
            "final_eval_mean_best_ckpt": float(final_eval["eval_mean"]),
            "final_eval_std_best_ckpt": float(final_eval["eval_std"]),
            "best_eval_mean_during_training": float(best_eval_mean if best_eval_mean > -1e17 else 0.0),
            "best_eval_episode": int(best_eval_episode),
            "spike_count": int(spike_count),
            "collapse_count": int(collapse_count),
        }

        with open(CFG.raw_csv, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=result.keys())
            writer.writeheader()
            writer.writerow(result)

        with open(CFG.summary_json, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print("\n================ FINAL ================")
        print(json.dumps(result, ensure_ascii=False, indent=2))
        print("RUN FINISHED")

    finally:
        try:
            env.close()
        except Exception:
            pass
        cleanup(device)


if __name__ == "__main__":
    main()

device       = cuda:0
use_amp      = False
cuda_name    = NVIDIA L40S
cuda_count   = 1
total_memory = 44.52 GB
[train] ep=00000/24000 (  0.0%) raw=-500.000 train=-474.246 recent20=-500.000 flicker=0.0000 noise=0.00000 lr=0.0000800 gpu=alloc=19.8MB reserved=52.0MB
[update] loss=417.0787 policy=-0.1841 value=596.1368 entropy=1.0986 kl=-0.0000 early_stop_kl=0 gpu=alloc=23.5MB reserved=92.0MB
[update] loss=438.7782 policy=0.0266 value=626.8351 entropy=1.0985 kl=0.0000 early_stop_kl=0 gpu=alloc=23.5MB reserved=92.0MB
[train] ep=00010/24000 (  0.0%) raw=-500.000 train=-469.392 recent20=-500.000 flicker=0.0000 noise=0.00000 lr=0.0000800 gpu=alloc=23.5MB reserved=92.0MB
[update] loss=401.9853 policy=-0.1881 value=574.5805 entropy=1.0985 kl=-0.0000 early_stop_kl=0 gpu=alloc=23.5MB reserved=92.0MB
[update] loss=426.8333 policy=0.0285 value=609.7683 entropy=1.0984 kl=-0.0001 early_stop_kl=0 gpu=alloc=23.5MB reserved=92.0MB
[update] loss=427.7475 policy=-0.0129 value=611.1334 entropy=1.0983 kl=-0.

In [1]:
import os
import json
import torch

# ==============================
# 설정
# ==============================
NAME = "acrobot_reliable_v4_stable"
SEED = 0
TARGET_EPISODES = 24000

CKPT_DIR = "./checkpoints_acrobot_v4"
LATEST_PATH = os.path.join(CKPT_DIR, f"{NAME}_seed{SEED}_latest.pt")
BEST_PATH = os.path.join(CKPT_DIR, f"{NAME}_seed{SEED}_best.pt")
SUMMARY_JSON = "./acrobot_v4_summary.json"
RAW_CSV = "./acrobot_v4_raw.csv"


def check_run_finished():
    result = {
        "latest_ckpt_exists": os.path.exists(LATEST_PATH),
        "best_ckpt_exists": os.path.exists(BEST_PATH),
        "summary_json_exists": os.path.exists(SUMMARY_JSON),
        "raw_csv_exists": os.path.exists(RAW_CSV),
        "episodes_completed": None,
        "target_episodes": TARGET_EPISODES,
        "finished_by_summary": False,
        "finished_by_ckpt": False,
        "run_finished": False,
    }

    # 1) summary json 기준 확인
    if os.path.exists(SUMMARY_JSON):
        try:
            with open(SUMMARY_JSON, "r", encoding="utf-8") as f:
                summary = json.load(f)

            completed = int(summary.get("episodes_completed", 0))
            result["episodes_completed"] = completed
            result["finished_by_summary"] = completed >= TARGET_EPISODES
            result["summary_content"] = summary
        except Exception as e:
            result["summary_error"] = str(e)

    # 2) latest checkpoint 기준 확인
    if os.path.exists(LATEST_PATH):
        try:
            ckpt = torch.load(LATEST_PATH, map_location="cpu")
            # 저장 시 episode_idx가 들어가므로 실제 완료 에피소드는 +1로 보는 게 자연스러움
            ckpt_episode = int(ckpt.get("episode", -1)) + 1
            result["ckpt_episode"] = ckpt_episode

            if result["episodes_completed"] is None:
                result["episodes_completed"] = ckpt_episode

            result["finished_by_ckpt"] = ckpt_episode >= TARGET_EPISODES
        except Exception as e:
            result["ckpt_error"] = str(e)

    # 3) 최종 판단
    # summary가 있으면 그걸 우선 신뢰, 없으면 ckpt 기준
    result["run_finished"] = result["finished_by_summary"] or result["finished_by_ckpt"]

    return result


info = check_run_finished()

print("=" * 60)
print(f"latest_ckpt_exists : {info['latest_ckpt_exists']}")
print(f"best_ckpt_exists   : {info['best_ckpt_exists']}")
print(f"summary_json_exists: {info['summary_json_exists']}")
print(f"raw_csv_exists     : {info['raw_csv_exists']}")
print(f"episodes_completed : {info['episodes_completed']}")
print(f"target_episodes    : {info['target_episodes']}")
print(f"finished_by_summary: {info['finished_by_summary']}")
print(f"finished_by_ckpt   : {info['finished_by_ckpt']}")
print(f"run_finished       : {info['run_finished']}")
print("=" * 60)

if info["run_finished"]:
    print("학습 끝남")
else:
    print("아직 안 끝남")

/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


latest_ckpt_exists : True
best_ckpt_exists   : True
summary_json_exists: True
raw_csv_exists     : True
episodes_completed : 24000
target_episodes    : 24000
finished_by_summary: True
finished_by_ckpt   : True
run_finished       : True
학습 끝남
